In [56]:
import os
from dotenv import load_dotenv
from urllib.request import urlopen
import json
import pandas as pd
from io import BytesIO
from google.cloud import storage

load_dotenv()

True

In [57]:
BASE_URL = os.getenv("BASE_URL", "").strip("/")
YEAR = int(os.getenv("YEAR", ""))
URL = f"{BASE_URL}/meetings?year={YEAR}"
GCS_BUCKET = os.getenv("GCS_BUCKET")
GCS_MEDAL_PUSH = os.getenv("GCS_MEDAL_PUSH")
BLOB_PATH = f"{GCS_MEDAL_PUSH}/season={YEAR}/meetings.parquet"

In [58]:
def fetch_data_from_api(URL):
    with urlopen(URL, timeout=120) as resp:
        raw = resp.read().decode("UTF-8")
    data = json.loads(raw)
    df = pd.DataFrame(data)
    df["ingested_at"] = pd.Timestamp.utcnow()
    return df

In [59]:
def upload_to_parquet(df, bucket, path, name):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    payload = buf.getvalue()

    blob = bucket.blob(path)
    blob.upload_from_file(
        BytesIO(payload),
        content_type="application/vnd.apache.parquet",
        size=len(payload),
    )

    df.to_parquet(f"../../data/bronze/season={YEAR}/{name}.parquet", index=False)


In [60]:
meetings_df = fetch_data_from_api(URL)

In [61]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

In [62]:
upload_to_parquet(meetings_df, bucket, BLOB_PATH, "meetings")